In [ ]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt?utm_source=chatgpt.com"

response = requests.get(url)
response.raise_for_status()

with open("training.txt", "w", encoding="utf-8") as file:
  file.write(response.text)

In [ ]:
with open("training.txt", "r", encoding="utf-8") as file:
  text = file.read()

text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [ ]:
import regex as re

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

In [ ]:
class BasicTokenizer():
  def __init__(self) -> None:
    self.merges={}

  def get_stats(self, ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
      counts[pair] = counts.get(pair, 0) + 1
    return counts

  def merge(self, ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
      if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
        new_ids.append(idx)
        i += 2
      else:
        new_ids.append(ids[i])
        i += 1
    return new_ids

  def train(self, text, vocab_size, verbose=True):
    num_merges = vocab_size - 256
    tokens = list(map(int, text.encode("utf-8")))
    for i in range(num_merges):
      stats = self.get_stats(tokens)
      if not stats:
        break
      pair = max(stats, key=stats.get)
      idx = 256 + i
      if verbose:
        print(f"merging {pair} into a new token {idx}")
      tokens = self.merge(tokens, pair, idx)
      self.merges[pair] = idx
    return tokens

  def encode(self, text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
      stats = self.get_stats(tokens)
      pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
      if pair not in self.merges:
        break
      idx = self.merges[pair]
      tokens = self.merge(tokens, pair, idx)
    return tokens

  def decode(self, ids):
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for (p0, p1), idx in self.merges.items():
      vocab[idx] = vocab[p0] + vocab[p1]
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

In [ ]:
tokenizer = BasicTokenizer()
tokens = tokenizer.train(text, 1000)
tokens

merging (101, 32) into a new token 256
merging (116, 104) into a new token 257
merging (116, 32) into a new token 258
merging (115, 32) into a new token 259
merging (100, 32) into a new token 260
merging (44, 32) into a new token 261
merging (111, 117) into a new token 262
merging (101, 114) into a new token 263
merging (105, 110) into a new token 264
merging (121, 32) into a new token 265
merging (97, 110) into a new token 266
merging (58, 10) into a new token 267
merging (111, 114) into a new token 268
merging (111, 32) into a new token 269
merging (101, 110) into a new token 270
merging (10, 10) into a new token 271
merging (97, 114) into a new token 272
merging (32, 257) into a new token 273
merging (111, 110) into a new token 274
merging (108, 108) into a new token 275
merging (104, 97) into a new token 276
merging (44, 10) into a new token 277
merging (46, 271) into a new token 278
merging (105, 259) into a new token 279
merging (101, 115) into a new token 280
merging (121, 262) 

[726,
 67,
 999,
 270,
 267,
 984,
 655,
 538,
 606,
 99,
 326,
 260,
 719,
 102,
 333,
 368,
 261,
 525,
 690,
 854,
 278,
 65,
 275,
 969,
 112,
 574,
 261,
 854,
 994,
 67,
 999,
 270,
 267,
 581,
 32,
 420,
 808,
 449,
 403,
 118,
 340,
 330,
 728,
 283,
 404,
 256,
 773,
 399,
 355,
 109,
 559,
 353,
 65,
 275,
 267,
 82,
 280,
 403,
 118,
 424,
 370,
 449,
 403,
 118,
 424,
 994,
 67,
 999,
 270,
 267,
 70,
 299,
 296,
 261,
 534,
 500,
 32,
 67,
 97,
 105,
 505,
 786,
 531,
 505,
 279,
 294,
 105,
 760,
 32,
 270,
 101,
 328,
 699,
 987,
 515,
 321,
 278,
 65,
 275,
 267,
 851,
 500,
 39,
 392,
 538,
 500,
 536,
 994,
 67,
 999,
 270,
 267,
 76,
 448,
 505,
 107,
 387,
 475,
 503,
 119,
 840,
 308,
 357,
 99,
 867,
 973,
 617,
 431,
 367,
 335,
 364,
 331,
 73,
 115,
 39,
 258,
 573,
 396,
 404,
 504,
 353,
 65,
 275,
 862,
 269,
 667,
 116,
 374,
 107,
 373,
 274,
 536,
 375,
 610,
 342,
 388,
 100,
 841,
 58,
 293,
 119,
 611,
 721,
 437,
 463,
 83,
 951,
 67,
 999,
 270,
 267

In [ ]:
tokenizer.encode("A")

[65]

In [ ]:
class RegexTokenizer():

    def __init__(self) -> None:
        self.merges = {}
        self.pattern = re.compile(GPT4_SPLIT_PATTERN)

    def get_stats(self, ids):
        counts = {}
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def merge(self, ids, pair, idx):
        new_ids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                new_ids.append(idx)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    def train(self, text, vocab_size, verbose=True):
        assert vocab_size >= 256

        self.merges = {}

        num_merges = vocab_size - 256

        chunks = self.pattern.findall(text)

        ids = [
            list(chunk.encode("utf-8"))
            for chunk in chunks
        ]

        for i in range(num_merges):

            stats = {}

            for chunk_ids in ids:
                chunk_stats = self.get_stats(chunk_ids)

                for pair, count in chunk_stats.items():
                    stats[pair] = stats.get(pair, 0) + count

            if not stats:
                break

            pair = max(stats, key=stats.get)

            idx = 256 + i

            if verbose:
                print(f"merging {pair} into a new token {idx}")

            ids = [
                self.merge(chunk_ids, pair, idx)
                for chunk_ids in ids
            ]

            self.merges[pair] = idx

        return [token for chunk_ids in ids for token in chunk_ids]

    def encode(self, text):
        chunks = self.pattern.findall(text)

        ids = []

        for chunk in chunks:
            tokens = list(chunk.encode("utf-8"))

            while len(tokens) >= 2:
                stats = self.get_stats(tokens)

                pair = min(
                    stats,
                    key=lambda p: self.merges.get(p, float("inf"))
                )

                if pair not in self.merges:
                    break

                idx = self.merges[pair]
                tokens = self.merge(tokens, pair, idx)

            ids.extend(tokens)

        return ids

    def decode(self, ids):
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        tokens = b"".join(vocab[idx] for idx in ids)

        text = tokens.decode("utf-8", errors="replace")

        return text


In [ ]:
tokenizer = RegexTokenizer()
tokens = tokenizer.train(text, 2000)


merging (32, 116) into a new token 256
merging (104, 101) into a new token 257
merging (32, 97) into a new token 258
merging (111, 117) into a new token 259
merging (32, 115) into a new token 260
merging (32, 109) into a new token 261
merging (105, 110) into a new token 262
merging (32, 119) into a new token 263
merging (114, 101) into a new token 264
merging (104, 97) into a new token 265
merging (58, 10) into a new token 266
merging (110, 100) into a new token 267
merging (256, 257) into a new token 268
merging (32, 98) into a new token 269
merging (105, 115) into a new token 270
merging (111, 114) into a new token 271
merging (10, 10) into a new token 272
merging (32, 102) into a new token 273
merging (101, 114) into a new token 274
merging (108, 108) into a new token 275
merging (105, 116) into a new token 276
merging (111, 110) into a new token 277
merging (44, 10) into a new token 278
merging (32, 100) into a new token 279
merging (32, 99) into a new token 280
merging (101, 115) 

In [ ]:
tokenizer.encode("I am a boy")

[73, 475, 258, 1305]

In [ ]:
tokenizer.decode([73, 475, 258, 1305])

'I am a boy'